# Census 2031 digital inclusion: reference analysis

**Decision:** where should research and alternative-contact interventions be prioritised before a digital-first Census?

**TL;DR:** Paper-first allocation explains 96.4% of variation in online completion across 328 authorities. Six authorities remain in the top 12 under all three fixed weighting schemes. The result supports a diverse research portfolio rather than treating one precise ranking as truth.

> This notebook uses the surviving derived reference output. It independently reproduces the regression, residual check and ranking intersection. It cannot regenerate the exact six-component Monte Carlo analysis because four raw demographic columns were not preserved.

## 1. Setup

The package code under `src/` is the canonical implementation; this notebook is a concise analytical walkthrough.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))

from ons_risk_index.reference import load_reference_scores, validate_reference
pd.set_option('display.max_rows', 15)

## 2. Load and validate the reference frame

Quality gates cover row count, entity uniqueness, percentage bounds and Monte Carlo-frequency bounds before analytical use.

In [ ]:
data_path = ROOT / 'data' / 'reference' / 'reference_scores.csv'
frame = load_reference_scores(data_path)

assert len(frame) == 328
assert frame['local_authority'].notna().all()
assert frame['local_authority'].is_unique
assert frame[['online_pct', 'paper_first_pct']].apply(lambda s: s.between(0, 100).all()).all()
assert frame['monte_carlo_frequency_reported'].between(0, 1).all()
frame.head()

## 3. Reproduce the allocation model

The model estimates online completion from the share of LSOAs allocated paper-first. The risk residual is expected minus actual online share, so a positive value indicates underperformance relative to allocation.

In [ ]:
checks, regression = validate_reference(frame)
pd.Series({
    'intercept': regression.intercept,
    'paper_first_slope': regression.slope,
    'r_squared': regression.r_squared,
    'observations': regression.n,
    'max_residual_difference': checks['max_absolute_residual_difference'],
}).to_frame('value')

In [ ]:
ax = frame.plot.scatter(
    x='paper_first_pct', y='online_pct', figsize=(9, 5.5),
    alpha=0.6, color='#2F75B5', title='Online completion vs paper-first allocation'
)
x = pd.Series([frame.paper_first_pct.min(), frame.paper_first_pct.max()])
ax.plot(x, regression.intercept + regression.slope * x, color='#E5A823', linewidth=2.5)
ax.set_xlabel('LSOAs allocated paper-first (%)')
ax.set_ylabel('Household returns completed online (%)')
ax.grid(alpha=0.2)
plt.show()

**Interpretation:** the very high R² means raw online completion is strongly shaped by the 2021 allocation design. It should not be read as a standalone measure of digital exclusion, and the association is not a causal estimate.

## 4. Inspect the evidence-led shortlist

In [ ]:
columns = [
    'local_authority', 'score_b', 'rank_b',
    'underperformance_reported', 'monte_carlo_frequency_reported'
]
frame.nsmallest(12, 'rank_b')[columns].reset_index(drop=True)

## 5. Check agreement across fixed schemes

In [ ]:
reported = set(frame.loc[frame['top_12_all_three'].eq('Yes'), 'local_authority'])
calculated = set.intersection(*(
    set(frame.nsmallest(12, f'rank_{scheme}')['local_authority'])
    for scheme in ('a', 'b', 'c')
))
assert reported == calculated
pd.DataFrame({'top_12_in_all_three_schemes': sorted(calculated)})

## 6. Review reported weighting robustness

In [ ]:
robust = frame.nlargest(12, 'monte_carlo_frequency_reported').sort_values('monte_carlo_frequency_reported')
ax = robust.plot.barh(
    x='local_authority', y='monte_carlo_frequency_reported',
    legend=False, figsize=(9, 6), color='#2F75B5',
    title='Reported top-12 frequency across 1,000 random weightings'
)
ax.set_xlabel('Frequency')
ax.set_ylabel('')
ax.grid(axis='x', alpha=0.2)
plt.show()

## 7. Decision interpretation and next steps

- Use a portfolio spanning **intensity, scale and underperformance**, not the first five rows of one ranking.
- Prioritise **first-contact** interventions because non-engagers cannot benefit from later form or digital-skills improvements.
- Combine area screening with resident research and trusted local organisations.
- Restore the four missing public demographic components, rerun the full pipeline and compare refreshed results before operational use.
- Treat local-authority scores as area-level signals, never as labels for individuals.